In [ ]:
"""Convert coin-quoted pairs to USDT-equivalent prices.

Fetches all spot tickers from Binance, then uses USDT pairs as
intermediaries to price non-USDT-quoted symbols in USDT.
No API keys required — uses public market data.
"""

In [ ]:
import ccxt

In [ ]:
from ccxt_pandas import CCXTPandasExchange

In [ ]:
binance = ccxt.binance()
exchange = CCXTPandasExchange(binance)

In [ ]:
# Load spot markets
markets = exchange.load_markets().query("type == 'spot'")
print("Quote currencies:", markets["quote"].unique())

In [ ]:
# Fetch tickers
tickers = exchange.fetch_tickers().dropna(subset=["last"])
tickers = tickers[["symbol", "last"]].merge(markets[["symbol", "base", "quote"]])

In [ ]:
# Get USDT reference prices
usdt_tickers = tickers.query("quote == 'USDT'")
usdt_tickers = usdt_tickers[["last", "symbol", "base"]].rename(
    columns={"last": "usdt_price", "symbol": "intermediate_symbol", "base": "quote"}
)

In [ ]:
# Merge to get USDT prices for coin-quoted pairs
tickers = tickers.merge(usdt_tickers)
tickers["usdt_price"] *= tickers["last"]

In [ ]:
print(tickers)
print("\nETH pairs:")
print(tickers.query("base == 'ETH'"))